# Resultados por país y compañía productora

Combina las películas estrenadas después de 2010 con sus países y compañías productoras y guarda el resultado en `gold/results_country_prod_company`. La tabla se registra después en `99. create_gold_tables`.

### Configuración

Carga las rutas de almacenamiento.

In [0]:
%run "../includes/configuration"

In [0]:
from pyspark.sql.functions import lit, col, current_timestamp

### Lectura de fuentes Silver

Lee desde Silver (Parquet): `movies`, `countries`, `production_countries`, `movies_companies` y `production_companies`.

In [0]:
movie_df = (
    spark.read
        .format("parquet")
        .load(f"{silver_folder_path}/movies")
)

In [0]:
countries_df = (
    spark.read
        .format("parquet")
        .load(f"{silver_folder_path}/countries")
)

In [0]:
production_countries_df = (
    spark.read
        .format("parquet")
        .load(f"{silver_folder_path}/production_countries")
)

In [0]:
movie_companies_df = (
    spark.read
        .format("parquet")
        .load(f"{silver_folder_path}/movies_companies")
)

In [0]:
production_companies_df = (
    spark.read
        .format("parquet")
        .load(f"{silver_folder_path}/production_companies")
)

### Construcción del resultado

Filtra películas posteriores a 2010, las une con países y compañías y agrega `created_date`.

In [0]:
results_country_prod_company = (
    movie_df.filter(movie_df["year_release_date"] > 2010)
        .join(production_countries_df,
              movie_df["movie_id"] == production_countries_df["movie_id"], "inner")
        .join(countries_df,
              production_countries_df["country_id"] == countries_df["country_id"], "inner")
        .join(movie_companies_df,
              movie_df["movie_id"] == movie_companies_df["movie_id"], "inner")
        .join(production_companies_df,
              movie_companies_df["company_id"] == production_companies_df["company_id"], "inner")
).select(movie_df["title"], movie_df["budget"], movie_df["revenue"],
         movie_df["duration_time"],movie_df["release_date"],
         countries_df["country_name"],
         production_companies_df["company_name"]
).withColumn("created_date", current_timestamp())

In [0]:
display(results_country_prod_company)

### Escritura en Gold

Guarda el resultado como Parquet en `gold/results_country_prod_company` (modo `overwrite`).

In [0]:
(
    results_country_prod_company
    .write
        .mode("overwrite")
        .format("parquet")
        .save(f"{gold_folder_path}/results_country_prod_company")
)